# 实验7.2 嵌入式轻量化大模型部署与测试实验

> **GitCode 昇腾 910B3 云沙箱 · Qwen1.5-0.5B-Chat · LoRA 微调模型部署与推理测试**

本实验是**实验4.2（LoRA 微调训练）**的延续。在训练实验中，我们使用 LoRA 方法对 Qwen1.5-0.5B-Chat 进行了参数高效微调，使其成为"华为昇腾 AI 助手"。训练产物由 `lab4.2_cann_sandbox_lora_train.ipynb` 产生，已保存在 `output/` 目录下，包含 `qwen_lora_finetuned`（最终权重）、`checkpoint-50`、`checkpoint-55`（训练中间检查点）等多个可选 LoRA 权重。本实验的任务是：**从 `output/` 目录中选择一个训练好的 LoRA 权重，合并部署到昇腾 910B3 NPU 平台上，并进行全面的推理测试，验证部署的正确性与微调的有效性。**

**运行环境**：`cann_9.0.0-py3.11-A2-arm` · `ASCEND 1*NPU 910B3` · `16vCPUs, 32GiB`

---

## 1. 实验概述

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>实验名称</strong></td>
<td style="text-align: left;">嵌入式轻量化大模型部署与测试实验</td>
</tr>
<tr>
<td style="text-align: left;"><strong>目标硬件</strong></td>
<td style="text-align: left;">昇腾 910B3 NPU（云沙箱）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN · PyTorch · torch_npu · Transformers · PEFT</td>
</tr>
<tr>
<td style="text-align: left;"><strong>基础模型</strong></td>
<td style="text-align: left;">Qwen1.5-0.5B-Chat（462M 参数）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>部署方式</strong></td>
<td style="text-align: left;">权重合并 → NPU 推理引擎 → 三层测试验证</td>
</tr>
</table>

**表格解读**：本实验是实验7.2（LoRA训练）的延续，目标是将训练得到的LoRA微调模型部署到昇腾910B3 NPU上并全面测试。部署采用权重合并方式——通过`merge_and_unload()`将LoRA增量合并回基座权重，生成可独立部署的完整模型，推理时零额外延迟。部署后通过三层测试验证：训练集内问题（验证学会了训练知识）、泛化问题（验证泛化能力）、边界用例（测试鲁棒性），并与未微调基座模型对比量化微调效果。

### 实验目标

- **知识目标**：理解 LoRA 权重合并（merge_and_unload）的原理与部署优势；理解大模型在昇腾 NPU 上的推理流程（KV Cache、预热、自回归生成）；理解部署测试的三层体系。
- **能力目标**：能够将 LoRA 微调模型合并并部署到昇腾 NPU；能够构建推理引擎并测量性能指标；能够设计对比实验验证微调与部署的有效性。
- **素养目标**：形成"训练→部署→测试→优化"的工程闭环意识。

### 1.1 输入文件、部署流程与输出文件详解

本节详细说明实验7.1（LoRA 训练）的输出如何作为本实验的输入，以及本实验最终产出的 `merged_model/` 与 `onnx_export/` 两个目录的文件内容与作用。

#### 一、输入文件：来自 lab7.1 的 `output/` 目录

lab7.1 使用 LoRA 方法对 Qwen1.5-0.5B-Chat 进行参数高效微调，训练产物保存在 `output/` 目录下，包含以下子目录：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">子目录</th>
<th style="text-align: left;">包含文件</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>qwen_lora_finetuned/</code></td>
<td style="text-align: left;"><code>adapter_config.json</code>、<code>adapter_model.safetensors</code>、<code>README.md</code></td>
<td style="text-align: left;">训练<strong>最终保存</strong>的 LoRA 权重，本实验默认选用它进行部署</td>
</tr>
<tr>
<td style="text-align: left;"><code>checkpoint-50/</code>、<code>checkpoint-55/</code></td>
<td style="text-align: left;">上述三个文件 + <code>optimizer.pt</code>、<code>scheduler.pt</code>、<code>rng_state.pth</code>、<code>scaler.pt</code>、<code>trainer_state.json</code>、<code>training_args.bin</code></td>
<td style="text-align: left;">训练<strong>中间检查点</strong>，可用于断点续训，也可选作部署权重</td>
</tr>
</table>

各文件作用：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>adapter_model.safetensors</code></td>
<td style="text-align: left;">训练得到的 LoRA 低秩矩阵 B 和 A 的权重（约 3 MB），是微调的核心产物</td>
</tr>
<tr>
<td style="text-align: left;"><code>adapter_config.json</code></td>
<td style="text-align: left;">LoRA 配置信息（秩 r=8、缩放系数 α=16、目标模块等），<code>PeftModel.from_pretrained</code> 据此加载适配器</td>
</tr>
<tr>
<td style="text-align: left;"><code>optimizer.pt</code></td>
<td style="text-align: left;">优化器状态（动量等），用于断点续训</td>
</tr>
<tr>
<td style="text-align: left;"><code>scheduler.pt</code></td>
<td style="text-align: left;">学习率调度器状态</td>
</tr>
<tr>
<td style="text-align: left;"><code>rng_state.pth</code></td>
<td style="text-align: left;">随机数生成器状态，保证续训可复现</td>
</tr>
<tr>
<td style="text-align: left;"><code>trainer_state.json</code></td>
<td style="text-align: left;">训练日志（loss 历史、学习率记录等）</td>
</tr>
<tr>
<td style="text-align: left;"><code>training_args.bin</code></td>
<td style="text-align: left;">训练超参数序列化</td>
</tr>
</table>

本实验在「3.2 初始化运行环境」中自动扫描 `output/` 下所有含 `adapter_model.safetensors` 的子目录，默认选用 `qwen_lora_finetuned`，也可切换为 `checkpoint-50` 或 `checkpoint-55`。

#### 二、部署流程

```
lab7.1 训练产物 (output/)                本实验部署流程
┌──────────────────────┐
│ qwen_lora_finetuned/ │
│  adapter_model.      │
│    safetensors (B·A) │────┐
│  adapter_config.json │    │
└──────────────────────┘    │
                            ↓
┌─────────────────────────────────────┐
│ 步骤一：加载基座 W₀ + LoRA 适配器 B·A │
│   merge_and_unload()                 │
│   W = W₀ + (α/r)·B·A                 │
└──────────────────┬──────────────────┘
                   ↓
┌─────────────────────────────────────┐
│ 输出 merged_model/                   │
│  （合并后的完整可部署模型）           │
└──────────────────┬──────────────────┘
                   ↓
┌─────────────────────────────────────┐
│ 步骤二：加载到昇腾 NPU 推理 + 预热    │
│ 步骤三：三层测试验证（训练集内/泛化/  │
│   边界）+ 基座对比                    │
└──────────────────┬──────────────────┘
                   ↓
┌─────────────────────────────────────┐
│ 步骤四：导出 ONNX（单步前向传播）      │
│   输出 onnx_export/                  │
└──────────────────┬──────────────────┘
                   ↓
            ATC 转换为昇腾 OM 格式
```

#### 三、输出文件：`merged_model/` 目录

通过 `merge_and_unload()` 将 LoRA 增量合并回基座权重后保存，是一个**自包含的标准 Transformers 模型目录**，可被 `AutoModelForCausalLM.from_pretrained('./merged_model')` 直接加载，无需 peft 库，推理时零额外延迟。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">大小</th>
<th style="text-align: left;">内容与作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>model.safetensors</code></td>
<td style="text-align: left;">~928 MB</td>
<td style="text-align: left;"><strong>合并后的完整模型权重</strong>（W₀ + LoRA 增量），FP16 格式，是部署推理的核心文件</td>
</tr>
<tr>
<td style="text-align: left;"><code>config.json</code></td>
<td style="text-align: left;">~1.3 KB</td>
<td style="text-align: left;">模型架构配置（层数、隐藏维度、注意力头数等），<code>from_pretrained</code> 据此构建网络结构</td>
</tr>
<tr>
<td style="text-align: left;"><code>generation_config.json</code></td>
<td style="text-align: left;">~0.2 KB</td>
<td style="text-align: left;">生成超参默认值（temperature、top_p 等），<code>model.generate</code> 时使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>tokenizer.json</code></td>
<td style="text-align: left;">~11 MB</td>
<td style="text-align: left;">分词器词表（BPE，约 15 万 token），文本与 token ID 互转</td>
</tr>
<tr>
<td style="text-align: left;"><code>tokenizer_config.json</code></td>
<td style="text-align: left;">~0.4 KB</td>
<td style="text-align: left;">分词器设置（特殊 token、chat template 引用等）</td>
</tr>
<tr>
<td style="text-align: left;"><code>chat_template.jinja</code></td>
<td style="text-align: left;">~0.3 KB</td>
<td style="text-align: left;">对话模板（Jinja2 格式），把 system/user/assistant 消息格式化为模型输入</td>
</tr>
</table>

合并后模型与基座模型大小相同，因为 LoRA 低秩增量已融入权重，不再有独立的 B/A 旁路矩阵。

#### 四、输出文件：`onnx_export/` 目录

将合并后模型的**单步前向传播**（input_ids, attention_mask → logits）导出为 ONNX 格式，用于通过 ATC 工具编译为昇腾专用 OM 格式。共 171 个文件：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">数量</th>
<th style="text-align: left;">内容与作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>qwen_merged.onnx</code></td>
<td style="text-align: left;">1 个（~1 MB）</td>
<td style="text-align: left;"><strong>ONNX 计算图结构</strong>（protobuf），定义算子拓扑、输入输出（input_ids/attention_mask → logits）、opset=14。体积小是因为大权重以外部数据形式存储</td>
</tr>
<tr>
<td style="text-align: left;"><code>model.model.embed_tokens.weight</code></td>
<td style="text-align: left;">1 个（~622 MB）</td>
<td style="text-align: left;"><strong>词嵌入矩阵的外部权重数据</strong>（token embedding 层权重）</td>
</tr>
<tr>
<td style="text-align: left;"><code>onnx__MatMul_*</code></td>
<td style="text-align: left;">169 个</td>
<td style="text-align: left;"><strong>各层 MatMul 算子的外部权重数据</strong>，对应 Transformer 中所有矩阵乘法的权重：Q/K/V 投影、输出投影、FFN 的 up/gate/down 投影、LM Head 等</td>
</tr>
</table>

**为什么拆成多文件**：ONNX protobuf 有 2GB 大小限制，`torch.onnx.export` 自动启用**外部数据格式（external data format）**，将大张量存为独立文件，`.onnx` 主文件只存图结构和小张量。

**作用**：

- `qwen_merged.onnx` + 外部权重文件一起构成完整的平台无关推理图
- 用 ATC 工具编译为昇腾 OM 格式：`atc --framework=5 --model='./onnx_export/qwen_merged.onnx' --soc_version=Ascend910B3 --input_shape='input_ids:1,32;attention_mask:1,32'`
- 实际部署时 ONNX/OM 负责**单步前向计算**，外层循环控制自回归生成（拼 token、KV Cache 管理）

> **注意**：LLM 完整自回归生成循环无法直接导出 ONNX，导出的是单步前向传播，这是标准做法。


## 2. 部署原理

### 2.1 为什么需要"合并 LoRA 权重"？

LoRA 训练时，模型权重为 **W = W₀ + (α/r)·B·A**：
- W₀ 是冻结的基座权重（不训练）
- B (d×r) 和 A (r×k) 是训练得到的低秩矩阵
- α/r 是缩放系数（本实验 α=16, r=8, 缩放=2）

**部署时**有两种选择：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方式</th>
<th style="text-align: left;">描述</th>
<th style="text-align: left;">优点</th>
<th style="text-align: left;">缺点</th>
</tr>
<tr>
<td style="text-align: left;"><strong>基座 + 适配器</strong></td>
<td style="text-align: left;">分别加载 W₀ 和 B·A</td>
<td style="text-align: left;">一个基座挂多个任务</td>
<td style="text-align: left;">推理有额外计算，需 peft 库</td>
</tr>
<tr>
<td style="text-align: left;"><strong>合并后模型</strong></td>
<td style="text-align: left;">W = W₀ + (α/r)·B·A</td>
<td style="text-align: left;">零额外延迟，无需 peft</td>
<td style="text-align: left;">每个任务一份完整模型</td>
</tr>
</table>

**表格解读**：LoRA部署有两种方式。"基座+适配器"方式分别加载基座权重W₀和LoRA增量B·A，推理时动态计算W = W₀ + (α/r)·B·A，优点是一个基座可挂载多个任务的LoRA适配器（共享基座节省存储），缺点是推理时有额外矩阵加法开销且依赖peft库。"合并后模型"方式通过`merge_and_unload()`预先将LoRA增量合并到基座权重中，生成标准模型，推理时无额外计算（零延迟）且无需peft库，缺点是每个任务需要一份完整模型。本实验采用合并方式，因为生产部署中推理性能和独立性更重要。

本实验采用**合并方式**（推荐用于生产部署），通过 `merge_and_unload()` 将 LoRA 增量合并回基座权重。

### 2.2 昇腾 NPU 推理流程

```
用户输入问题
    ↓
Chat Template 格式化 (system + user → input_ids)
    ↓
Tokenize (文本 → 数字 ID 序列)
    ↓
NPU 前向传播 (input_ids → logits)    ← torch_npu 调用 CANN 算子
    ↓
采样/贪心解码 (logits → next_token)
    ↓
拼接 next_token 到 input_ids
    ↓
重复前向传播 (使用 KV Cache 加速)    ← 自回归生成循环
    ↓
遇到 EOS 或达到 max_new_tokens → 停止
    ↓
Detokenize (数字 ID → 文本) → 输出回答
```

### 2.3 KV Cache 的作用

- **无 KV Cache**：每生成一个 token，需对所有前文重新计算注意力 → O(n²) 复杂度
- **有 KV Cache**：缓存已计算的 Key/Value，每步只算新 token 的 Q/K/V → O(n) per step

### 2.4 预热（Warmup）的必要性

首次推理时，PyTorch/torch_npu 需要：
1. 编译计算图（JIT/trace）
2. 分配显存
3. 初始化 NPU 内核

这些一次性开销使首次推理显著慢于后续推理。**预热后测量才能反映真实性能。**

## 3. 环境初始化

### 3.1 安装依赖库

> 本实验需要 `transformers` 和 `peft` 库。若未安装，下方代码会自动安装。

In [ ]:
import subprocess
import sys

def ensure_package(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f'[OK] {package} 已安装')
    except ImportError:
        print(f'[安装] {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f'[完成] {package} 安装成功')

# 修复: 部分昇腾环境中 torchaudio 的 C 扩展与当前 torch 版本不匹配，导入时会报
# OSError: undefined symbol: torch_library_impl，连锁导致 transformers/peft 导入失败。
# NLP 任务不需要 torchaudio，先尝试卸载；若卸载失败则在 sys.modules 中屏蔽。
try:
    import importlib
    importlib.import_module('torchaudio')
except Exception:
    for _m in [m for m in list(sys.modules) if m == 'torchaudio' or m.startswith('torchaudio.')]:
        sys.modules.pop(_m, None)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
    try:
        importlib.import_module('torchaudio')
    except Exception:
        sys.modules['torchaudio'] = None
        print('[修复] 已屏蔽不兼容的 torchaudio（NLP 任务不需要）')
    else:
        print('[修复] 已卸载不兼容的 torchaudio（NLP 任务不需要）')

ensure_package('transformers')
ensure_package('peft')
print('依赖检查完成!')

### 3.2 初始化运行环境

In [ ]:
!pip install "transformers==4.39.3" -q
import os
import sys
import time
import json

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
try:
    import torch_npu
    NPU_AVAILABLE = torch.npu.is_available()
except ImportError:
    NPU_AVAILABLE = False
    print('[警告] torch_npu 未安装，将使用 CPU')

from transformers import AutoModelForCausalLM, AutoTokenizer

if NPU_AVAILABLE:
    device = torch.device('npu:0')
    torch.npu.set_device(0)
    print(f'设备: 昇腾 NPU ({device})')
else:
    device = torch.device('cpu')
    print(f'设备: CPU (NPU 不可用)')

MODEL_NAME = 'Qwen/Qwen1.5-0.5B-Chat'
OUTPUT_DIR = './output'
MERGED_PATH = './merged_model'
SYSTEM_PROMPT = '你是华为昇腾AI助手，请简洁准确地回答问题。'

# 扫描 output 目录下可用的 LoRA 权重（含 adapter_model.safetensors 的子目录）
# 训练产物由 test7_2-finish/02_cann_sandbox_lora_train.ipynb 生成，包含
# qwen_lora_finetuned（最终权重）与 checkpoint-*（中间检查点）等可选项
available_lora = []
if os.path.isdir(OUTPUT_DIR):
    for name in sorted(os.listdir(OUTPUT_DIR)):
        sub = os.path.join(OUTPUT_DIR, name)
        if os.path.isdir(sub) and os.path.exists(os.path.join(sub, 'adapter_model.safetensors')):
            available_lora.append(name)

# 选择部署用的 LoRA 权重：默认 qwen_lora_finetuned，否则取最新 checkpoint
LORA_SUBDIR = 'qwen_lora_finetuned' if 'qwen_lora_finetuned' in available_lora else (available_lora[-1] if available_lora else 'qwen_lora_finetuned')
LORA_PATH = os.path.join(OUTPUT_DIR, LORA_SUBDIR)

print(f'基座模型: {MODEL_NAME}')
print(f'output 目录可用 LoRA 权重: {available_lora if available_lora else "(无)"}')
print(f'当前选用 LoRA 权重: {LORA_PATH}')
print(f'合并输出: {MERGED_PATH}')
print(f'如需切换模型，请修改 LORA_SUBDIR 后重新运行本单元格')

**代码说明与预期结果**：

- **环境初始化**：设置HF镜像、检测NPU、导入transformers、定义关键路径常量。代码会自动扫描 `output/` 目录下所有含 `adapter_model.safetensors` 的子目录作为可选 LoRA 权重，默认选用 `qwen_lora_finetuned`（训练最终保存的权重），也可改为 `checkpoint-50` 或 `checkpoint-55` 等中间检查点。`MERGED_PATH` 是合并后模型的输出目录。
- **预期输出**：打印设备信息（NPU或CPU）、基座模型名、可用 LoRA 权重列表、当前选用的 LoRA 权重路径、合并输出路径。
- **前置条件**：`output/` 目录需已存在训练产物，由 `test7_2-finish/02_cann_sandbox_lora_train.ipynb` 训练生成（含 `qwen_lora_finetuned/` 等子目录）。

> **注意**：若报 `ModuleNotFoundError: No module named 'transformers'`，请先运行上方「3.1 安装依赖库」单元格。

## 4. 步骤一：合并 LoRA 权重

**目标**：将训练得到的 LoRA 增量权重合并回基座模型，生成可独立部署的完整模型。

**流程**：
1. 加载基座模型 Qwen1.5-0.5B-Chat（FP16）
2. 加载 LoRA 适配器（`output/` 下选定的权重目录，如 `qwen_lora_finetuned/`）
3. 调用 `merge_and_unload()` 合并权重：W = W₀ + (α/r)·B·A
4. 保存合并后模型到 `merged_model/`
5. 快速验证合并后模型推理正常

In [ ]:
# 检查 LoRA 权重是否存在
adapter_file = os.path.join(LORA_PATH, 'adapter_model.safetensors')
if not os.path.exists(adapter_file):
    print(f'[警告] 未找到 LoRA 权重: {adapter_file}')
    print('请先运行 test7_2-finish/02_cann_sandbox_lora_train.ipynb 完成训练')
    print('或将训练产物（含 adapter_model.safetensors）复制到 ./output/ 目录')
else:
    print(f'LoRA权重文件: {os.path.getsize(adapter_file) / 1024:.1f} KB')

**代码说明**：检查当前选用的 LoRA 权重文件 `adapter_model.safetensors` 是否存在。若不存在则提示先运行 `test7_2-finish/02_cann_sandbox_lora_train.ipynb` 完成训练，或将训练产物复制到 `./output/` 目录。预期输出权重文件大小约几 KB 到几 MB。

> **注意**：若报 `NameError: name 'LORA_PATH' is not defined`，请先运行「3.1 安装依赖库」→「3.2 初始化运行环境」单元格。

In [ ]:
from peft import PeftModel
import shutil

if os.path.exists(adapter_file):
    print('[1/5] 加载基座模型 Qwen1.5-0.5B-Chat...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    print('      基座模型已加载 (FP16)')

    print('[2/5] 加载 LoRA 适配器权重...')
    peft_model = PeftModel.from_pretrained(base_model, LORA_PATH)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f'      总参数量: {total_params / 1e6:.1f}M')

    print('[3/5] 合并 LoRA 权重到基座模型...')
    print('      W = W₀ + (α/r)·B·A')
    start_time = time.time()
    merged_model = peft_model.merge_and_unload()
    merge_time = time.time() - start_time
    print(f'      合并完成! 耗时: {merge_time:.2f} 秒')

    merged_params = sum(p.numel() for p in merged_model.parameters())
    print(f'      合并后参数量: {merged_params / 1e6:.1f}M (应与基座一致)')
else:
    print('跳过合并步骤，请先完成训练实验')

**代码说明与预期结果**：

- **合并流程**：加载基座模型→用`PeftModel.from_pretrained`加载LoRA适配器→调用`merge_and_unload()`将LoRA增量(α/r)·B·A加到基座权重W₀上，得到合并后的标准模型。
- **预期输出**：总参数量约462M（基座+LoRA），合并后参数量约462M（与基座一致，因为LoRA增量已合并进基座权重，不再有独立旁路）。合并耗时约0.5秒。
- **为什么参数量不变**：合并操作W = W₀ + (α/r)·B·A将低秩增量加到原始权重中，消除了B和A矩阵，模型结构恢复为标准Transformer。

> **注意**：若报 `ModuleNotFoundError: No module named 'peft'`，请先运行「3.1 安装依赖库」单元格。

In [ ]:
if os.path.exists(adapter_file):
    print('[4/5] 保存合并后的完整模型...')
    if os.path.exists(MERGED_PATH):
        shutil.rmtree(MERGED_PATH)
    os.makedirs(MERGED_PATH, exist_ok=True)

    merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_PATH)

    print(f'      模型已保存至: {MERGED_PATH}')
    total_size = 0
    for f in sorted(os.listdir(MERGED_PATH)):
        size = os.path.getsize(os.path.join(MERGED_PATH, f))
        total_size += size
        print(f'        {f}: {size / 1024:.1f} KB')
    print(f'      总大小: {total_size / 1024 / 1024:.1f} MB')

**代码说明与预期结果**：将合并后模型保存到`./merged_model/`目录。预期生成`model.safetensors`（约924MB，包含合并后的完整权重）和`config.json`、`tokenizer`文件。总大小约924MB——合并后模型与基座模型大小相同，因为LoRA增量已融入权重中。

In [ ]:
if os.path.exists(MERGED_PATH):
    print('[5/5] 合并后模型推理验证...')
    merged_model = merged_model.to(device)
    merged_model.eval()

    test_q = '什么是昇腾910？'
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': test_q},
    ]
    _inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', tokenize=True,
    )
    input_ids = _inputs if hasattr(_inputs, 'shape') else _inputs['input_ids']
    input_ids = input_ids.to(device)
    with torch.no_grad():
        outputs = merged_model.generate(
            input_ids, max_new_tokens=100,
            do_sample=False, top_p=0.9, temperature=0.1,
        )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print(f'  问: {test_q}')
    print(f'  答: {response[:150]}')

**代码说明与预期结果**：将合并后模型搬到NPU并做推理验证。对"什么是昇腾910？"提问，预期回答包含昇腾910的关键信息（如256TFLOPS算力、达芬奇架构等），与微调后效果一致。这验证了合并操作正确保留了微调知识。

## 5. 步骤二：昇腾 NPU 部署推理

**目标**：将合并后模型加载到昇腾 NPU，构建推理引擎，支持多种推理模式。

### 5.1 NPU 推理引擎类

封装模型加载、预热、推理、性能统计的完整流程。

In [ ]:
class NPUInferenceEngine:
    """昇腾 NPU 大模型推理引擎"""

    def __init__(self, model_path, device, dtype=torch.float16):
        self.device = device
        self.dtype = dtype
        self.model_path = model_path

        print(f'加载模型与分词器...')
        print(f'  路径: {model_path}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=dtype,
        ).to(device)
        self.model.eval()

        total_params = sum(p.numel() for p in self.model.parameters())
        print(f'  参数量: {total_params / 1e6:.1f}M')
        print(f'  数据类型: {dtype}')

        self.inference_count = 0
        self.total_tokens = 0
        self.total_time = 0.0

    def warmup(self):
        """预热推理引擎"""
        print(f'预热推理引擎 (warmup)...')
        dummy_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': '你好'},
        ]
        _inputs = self.tokenizer.apply_chat_template(
            dummy_messages, add_generation_prompt=True,
            return_tensors='pt', tokenize=True,
        )
        input_ids = _inputs if hasattr(_inputs, 'shape') else _inputs['input_ids']
        input_ids = input_ids.to(self.device)
        with torch.no_grad():
            _ = self.model.generate(input_ids, max_new_tokens=8, do_sample=False)
        print(f'  预热完成')

    def infer(self, question, max_new_tokens=128, do_sample=False):
        """执行单次推理"""
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': question},
        ]
        _inputs = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors='pt', tokenize=True,
        )
        input_ids = _inputs if hasattr(_inputs, 'shape') else _inputs['input_ids']
        input_ids = input_ids.to(self.device)
        input_len = input_ids.shape[-1]

        torch.npu.synchronize() if NPU_AVAILABLE else None
        start_time = time.time()

        with torch.no_grad():
            outputs = self.model.generate(
                input_ids, max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        torch.npu.synchronize() if NPU_AVAILABLE else None
        latency = time.time() - start_time

        new_tokens = outputs.shape[-1] - input_len
        response = self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

        self.inference_count += 1
        self.total_tokens += new_tokens
        self.total_time += latency

        return {
            'response': response,
            'new_tokens': new_tokens,
            'latency': latency,
            'tokens_per_sec': new_tokens / latency if latency > 0 else 0,
        }

    def print_stats(self):
        """打印推理性能统计"""
        print(f'\n  推理统计:')
        print(f'    总推理次数: {self.inference_count}')
        print(f'    总生成 token: {self.total_tokens}')
        print(f'    总耗时: {self.total_time:.2f} 秒')
        if self.total_time > 0:
            print(f'    平均吞吐量: {self.total_tokens / self.total_time:.1f} token/s')

print('NPUInferenceEngine 类定义完成!')

In [ ]:
if os.path.exists(MERGED_PATH):
    engine = NPUInferenceEngine(MERGED_PATH, device)
    engine.warmup()
else:
    print('合并模型不存在，请先完成步骤一')

**代码说明与预期结果**：用合并后模型实例化`NPUInferenceEngine`并执行预热（warmup）。预热时模型做一次短推理（max_new_tokens=8），触发计算图编译和显存分配。预期打印模型加载信息和"预热完成"。预热后后续推理的性能测量才准确——首次推理因编译开销显著偏慢。

> **注意**：若报 `NameError: name 'MERGED_PATH' is not defined`，请先运行「3.1 安装依赖库」→「3.2 初始化运行环境」单元格。

### 5.2 性能基准测试

In [ ]:
if os.path.exists(MERGED_PATH):
    benchmark_questions = [
        '什么是昇腾910？',
        '什么是LoRA？',
        'MindSpore有什么特点？',
        '什么是CANN？',
        '什么是达芬奇架构？',
    ]

    print('=== 性能基准测试 ===')
    print(f'  测试问题数: {len(benchmark_questions)}')
    print()

    results = []
    for i, q in enumerate(benchmark_questions):
        result = engine.infer(q, max_new_tokens=128)
        result['question'] = q
        result['index'] = i
        results.append(result)

    print(f"  {'序号':<6}{'问题':<20}{'Token数':<10}{'耗时(s)':<12}{'吞吐(tok/s)':<14}")
    print('  ' + '-' * 60)
    for r in results:
        q_short = r['question'][:18]
        print(f"  {r['index']+1:<6}{q_short:<20}{r['new_tokens']:<10}{r['latency']:<12.3f}{r['tokens_per_sec']:<14.1f}")

    print('  ' + '-' * 60)
    avg_latency = sum(r['latency'] for r in results) / len(results)
    avg_throughput = sum(r['tokens_per_sec'] for r in results) / len(results)
    print(f'  平均耗时: {avg_latency:.3f} 秒/问')
    print(f'  平均吞吐: {avg_throughput:.1f} token/s')

    engine.print_stats()

**代码说明与预期结果**：用5个昇腾相关问题做性能基准测试，每个问题生成128个token。输出表格显示每题的token数、耗时和吞吐量。预期平均耗时约0.2秒/问，平均吞吐约190 token/s（昇腾910B3 NPU）。`engine.print_stats()`打印累计统计。这量化了部署后的推理性能。

## 6. 步骤三：全面测试验证

**三层测试体系**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">测试层</th>
<th style="text-align: left;">问题来源</th>
<th style="text-align: left;">目的</th>
</tr>
<tr>
<td style="text-align: left;"><strong>训练集内</strong></td>
<td style="text-align: left;">训练数据中的问题</td>
<td style="text-align: left;">验证模型是否学会了训练知识</td>
</tr>
<tr>
<td style="text-align: left;"><strong>泛化测试</strong></td>
<td style="text-align: left;">训练集外相关问题</td>
<td style="text-align: left;">验证模型泛化能力</td>
</tr>
<tr>
<td style="text-align: left;"><strong>边界用例</strong></td>
<td style="text-align: left;">非领域问题</td>
<td style="text-align: left;">测试模型鲁棒性</td>
</tr>
</table>

**评估方法**：关键词命中率——每个问题预定义期望关键词，检查回答是否包含关键词，命中率 ≥ 50% 判定为通过。

**表格解读**：三层测试体系从不同维度验证模型质量。训练集内测试用训练数据中的问题验证模型是否记住了训练知识（微调后命中率应显著高于基座）。泛化测试用训练集外但领域相关的问题（如"什么是梯度下降？"虽在训练集中，但"什么是Transformer？"的期望关键词可能不同）验证模型是否真正理解而非死记硬背。边界用例用非领域问题（如"1+1等于几？"）测试模型是否因微调而丧失通用能力（过拟合检测）。关键词命中率方法简单有效——预定义每个问题的期望关键词（如"昇腾910"对应["昇腾","华为","训练","256"]），检查回答包含多少关键词，命中率≥50%判为通过。

In [ ]:
test_questions = {
    'in_distribution': [
        {'question': '什么是昇腾910？', 'expected_keyword': ['昇腾', '华为', '训练', '256']},
        {'question': '什么是LoRA？', 'expected_keyword': ['低秩', '参数', '微调']},
        {'question': 'MindSpore有什么特点？', 'expected_keyword': ['框架', '全场景', '易开发']},
        {'question': '什么是CANN？', 'expected_keyword': ['异构', '计算', '算子']},
        {'question': '什么是达芬奇架构？', 'expected_keyword': ['架构', 'Cube', '矩阵']},
        {'question': '什么是模型微调？', 'expected_keyword': ['预训练', '微调', '特定']},
    ],
    'out_of_distribution': [
        {'question': '什么是梯度下降？', 'expected_keyword': ['梯度', '优化', '损失']},
        {'question': '什么是Transformer？', 'expected_keyword': ['注意力', '自注意力', '并行']},
        {'question': '什么是注意力机制？', 'expected_keyword': ['注意力', '查询', '键', '值']},
        {'question': '什么是过拟合？', 'expected_keyword': ['训练集', '泛化', '正则']},
        {'question': '昇腾910和310有什么区别？', 'expected_keyword': ['训练', '推理', '910', '310']},
    ],
    'edge_cases': [
        {'question': '你好，请介绍一下你自己。', 'expected_keyword': ['昇腾', '助手']},
        {'question': '1+1等于几？', 'expected_keyword': ['2', '二', '等于']},
        {'question': '请用一句话说明什么是深度学习。', 'expected_keyword': ['神经网络', '学习', '特征']},
    ],
}

def check_keywords(response, expected_keywords):
    hits = [kw for kw in expected_keywords if kw.lower() in response.lower()]
    hit_rate = len(hits) / len(expected_keywords) if expected_keywords else 0
    return hit_rate, hits

def ask_test(question, model, tokenizer, device, max_new_tokens=128):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    _inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', tokenize=True,
    )
    input_ids = _inputs if hasattr(_inputs, 'shape') else _inputs['input_ids']
    input_ids = input_ids.to(device)
    input_len = input_ids.shape[-1]
    torch.npu.synchronize() if NPU_AVAILABLE else None
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    torch.npu.synchronize() if NPU_AVAILABLE else None
    latency = time.time() - start
    new_tokens = outputs.shape[-1] - input_len
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return {'response': response, 'new_tokens': new_tokens, 'latency': latency,
            'tokens_per_sec': new_tokens / latency if latency > 0 else 0}

print('测试函数定义完成!')

In [ ]:
if os.path.exists(MERGED_PATH):
    # 加载基座模型用于对比
    print('加载基座模型（未微调，用于对比）...')
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16
    ).to(device)
    base_model.eval()

    # 预热
    print('预热模型...')
    _ = ask_test('你好', merged_model, tokenizer, device, max_new_tokens=8)
    _ = ask_test('你好', base_model, tokenizer, device, max_new_tokens=8)
    print('预热完成')
else:
    print('合并模型不存在，请先完成步骤一')

**代码说明**：加载未微调的基座模型用于对比测试，并对微调后模型和基座模型分别预热。这样后续测试可以公平对比微调前后效果差异。

In [ ]:
if os.path.exists(MERGED_PATH):
    all_results = []

    def run_test_group(group_name, questions, model, tokenizer, device, model_label):
        print(f'\n  【{group_name}】({model_label})')
        print('  ' + '-' * 66)
        group_results = []
        for item in questions:
            q = item['question']
            expected_kw = item.get('expected_keyword', [])
            result = ask_test(q, model, tokenizer, device)
            hit_rate, hits = check_keywords(result['response'], expected_kw)
            result['question'] = q
            result['hit_rate'] = hit_rate
            result['model_label'] = model_label
            result['group'] = group_name
            group_results.append(result)
            status = 'PASS' if hit_rate >= 0.5 else 'FAIL'
            print(f'  [{status}] {q}')
            print(f'        回答: {result["response"][:80]}')
            print(f'        关键词命中: {len(hits)}/{len(expected_kw)} ({hit_rate:.0%})')
        return group_results

    print('========== 测试一：微调后模型 vs 训练集内问题 ==========')
    r1 = run_test_group('训练集内', test_questions['in_distribution'], merged_model, tokenizer, device, '微调后')
    all_results.extend(r1)

    print('\n========== 测试二：微调后模型 vs 泛化问题 ==========')
    r2 = run_test_group('泛化测试', test_questions['out_of_distribution'], merged_model, tokenizer, device, '微调后')
    all_results.extend(r2)

    print('\n========== 测试三：微调后模型 vs 边界用例 ==========')
    r3 = run_test_group('边界用例', test_questions['edge_cases'], merged_model, tokenizer, device, '微调后')
    all_results.extend(r3)

**代码说明与预期结果**：对微调后模型执行三层测试。测试一用训练集内问题（如"什么是昇腾910？"），预期PASS率高（微调后学会了训练知识）。测试二用泛化问题（如"什么是Transformer？"），验证模型是否真正理解而非死记。测试三用边界用例（如"1+1等于几？"），验证微调未破坏通用能力。每题打印PASS/FAIL、回答摘要和关键词命中率。

In [ ]:
if os.path.exists(MERGED_PATH):
    print('========== 测试四：基座模型（未微调）对比 ==========')
    r4 = run_test_group('训练集内', test_questions['in_distribution'], base_model, tokenizer, device, '基座(未微调)')
    all_results.extend(r4)
    r5 = run_test_group('泛化测试', test_questions['out_of_distribution'], base_model, tokenizer, device, '基座(未微调)')
    all_results.extend(r5)

**代码说明与预期结果**：对未微调基座模型执行相同测试作为对照。预期基座模型在训练集内问题上PASS率显著低于微调后模型（因缺乏昇腾领域知识），这量化证明了LoRA微调的有效性。对比微调后与基座的关键词命中率差异即为微调提升幅度。

### 6.1 测试报告总结

In [ ]:
if os.path.exists(MERGED_PATH):
    def summarize(results, label):
        subset = [r for r in results if r['model_label'] == label]
        if not subset:
            return {}
        avg_hit = sum(r['hit_rate'] for r in subset) / len(subset)
        avg_latency = sum(r['latency'] for r in subset) / len(subset)
        avg_throughput = sum(r['tokens_per_sec'] for r in subset) / len(subset)
        pass_count = sum(1 for r in subset if r['hit_rate'] >= 0.5)
        return {
            'question_count': len(subset),
            'avg_hit_rate': avg_hit,
            'pass_count': pass_count,
            'pass_rate': pass_count / len(subset),
            'avg_latency': avg_latency,
            'avg_throughput': avg_throughput,
        }

    s_ft = summarize(all_results, '微调后')
    s_base = summarize(all_results, '基座(未微调)')

    print('=' * 70)
    print('  部署测试报告总结')
    print('=' * 70)
    print('\n  一、微调前后效果对比')
    print('  ' + '-' * 66)
    print(f"  {'指标':<20}{'微调后模型':<25}{'基座(未微调)':<25}")
    print('  ' + '-' * 66)
    if s_ft and s_base:
        print(f"  {'问题数':<20}{s_ft['question_count']:<25}{s_base['question_count']:<25}")
        print(f"  {'平均关键词命中率':<20}{s_ft['avg_hit_rate']:.1%}{'':<15}{s_base['avg_hit_rate']:.1%}")
        print(f"  {'通过率(≥50%命中)':<20}{s_ft['pass_rate']:.1%}{'':<15}{s_base['pass_rate']:.1%}")
        print(f"  {'平均耗时(s)':<20}{s_ft['avg_latency']:.3f}{'':<15}{s_base['avg_latency']:.3f}")
        print(f"  {'平均吞吐(tok/s)':<20}{s_ft['avg_throughput']:.1f}{'':<15}{s_base['avg_throughput']:.1f}")

        print('\n  三、测试结论')
        print('  ' + '-' * 66)
        improvement = s_ft['avg_hit_rate'] - s_base['avg_hit_rate']
        if improvement > 0.1:
            print(f"  ✓ 微调有效: 关键词命中率提升 {improvement:.1%} ({s_base['avg_hit_rate']:.1%} → {s_ft['avg_hit_rate']:.1%})")
        elif improvement > 0:
            print(f"  △ 微调有一定效果: 关键词命中率提升 {improvement:.1%}")
        else:
            print(f"  ✗ 微调效果不明显: 关键词命中率变化 {improvement:.1%}")
        print(f'  ✓ 部署成功: 模型在 {device} 上正常推理')
        print(f"  ✓ 平均吞吐量: {s_ft['avg_throughput']:.1f} token/s")

    # 保存测试报告
    report = {
        'test_time': time.strftime('%Y-%m-%d %H:%M:%S'),
        'device': str(device),
        'summary': {'微调后': s_ft, '基座': s_base},
    }
    with open('test_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2, default=str)
    print(f'\n  测试报告已保存: test_report.json')

**代码说明与预期结果**：

- **报告总结**：汇总微调后模型与基座模型的关键词命中率、通过率、平均耗时和吞吐量，生成对比表格。根据命中率提升幅度判定微调效果（>10%为有效，>0为有一定效果，≤0为不明显）。
- **预期输出**：微调后模型平均命中率约65%，基座约20%，提升约45%。通过率微调后显著高于基座。推理吞吐量约190 token/s。测试报告保存至`test_report.json`。
- **结论判定**：若微调后命中率显著高于基座（提升>10%），说明LoRA微调有效且部署正确；若两者接近，可能训练不充分或数据质量不足。

## 7. 步骤四（进阶）：导出 ONNX 模型

> LLM 的完整自回归生成循环无法直接导出为 ONNX。本步骤导出的是"单步前向传播"（input_ids, attention_mask → logits），实际部署中 ONNX/OM 负责单步计算，外层循环控制生成。

**流程**：
1. 加载合并后模型（CPU, FP32）
2. 包装模型，仅输出 logits
3. 构造 dummy 输入（batch=1, seq_len=32）
4. 调用 `torch.onnx.export` 导出单步前向传播
5. 验证 ONNX 模型结构

In [ ]:
import os
import torch

ONNX_DIR = './onnx_export'
ONNX_PATH = os.path.join(ONNX_DIR, 'qwen_merged.onnx')
SEQ_LEN = 32  # 固定序列长度，需与 ATC 转换的 --input_shape 一致

if not os.path.exists(MERGED_PATH):
    print('[跳过] 合并模型不存在，请先完成步骤一')
else:
    os.makedirs(ONNX_DIR, exist_ok=True)

    # [1/4] 加载合并后模型（CPU, FP32 — ONNX 为平台无关格式）
    print('[1/4] 加载合并后模型 (CPU, FP32)...')
    onnx_model = AutoModelForCausalLM.from_pretrained(
        MERGED_PATH, torch_dtype=torch.float32
    )
    onnx_model.eval()
    print(f'      模型已加载自: {MERGED_PATH}')

    # 包装模型，仅输出 logits
    class LogitsOnlyWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, input_ids, attention_mask):
            return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

    wrapped = LogitsOnlyWrapper(onnx_model)

    # [2/4] 构造 dummy 输入
    print(f'[2/4] 构造 dummy 输入 (batch=1, seq_len={SEQ_LEN})...')
    dummy_input_ids = torch.zeros((1, SEQ_LEN), dtype=torch.long)
    dummy_attention_mask = torch.ones((1, SEQ_LEN), dtype=torch.long)

    # [3/4] 导出 ONNX（单步前向传播）
    print('[3/4] 导出 ONNX 模型 (input_ids, attention_mask → logits)...')
    with torch.no_grad():
        torch.onnx.export(
            wrapped,
            (dummy_input_ids, dummy_attention_mask),
            ONNX_PATH,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            opset_version=14,
            do_constant_folding=True,
        )
    onnx_size = os.path.getsize(ONNX_PATH) / 1024 / 1024
    print(f'      ONNX 已保存: {ONNX_PATH} ({onnx_size:.1f} MB)')

    # [4/4] 验证 ONNX 模型
    print('[4/4] 验证 ONNX 模型...')
    try:
        import onnx
        proto = onnx.load(ONNX_PATH)
        onnx.checker.check_model(proto)
        print(f'      输入: {[inp.name for inp in proto.graph.input]}')
        print(f'      输出: {[out.name for out in proto.graph.output]}')
        print('      ONNX 模型验证通过!')
    except ImportError:
        print('      [提示] onnx 库未安装，跳过验证 (pip install onnx)')
    except Exception as e:
        print(f'      [警告] 验证失败: {e}')

    print(f'\n导出完成! 下一步使用 ATC 转换为昇腾 OM 格式。')

**代码说明与预期结果**：

- **模型加载**：ONNX 为平台无关格式，导出时使用 CPU + FP32，避免 NPU 专有算子影响 tracing。`LogitsOnlyWrapper` 仅保留 `logits` 输出，简化 ONNX 计算图。
- **固定形状**：`SEQ_LEN=32` 与后续 ATC `--input_shape` 保持一致，ATC 据此优化计算图。如需支持其他长度，修改 `SEQ_LEN` 后重新运行。
- **预期输出**：ONNX 文件约 900 MB（与模型权重相当），验证打印输入 `[input_ids, attention_mask]`、输出 `[logits]`。
- **opset_version=14**：兼容大多数 ONNX Runtime 和 ATC 版本。

> **注意**：若报 `ModuleNotFoundError: No module named 'onnx'`，验证步骤会自动跳过，不影响导出。若导出时报 `RuntimeError`，可尝试减小 `SEQ_LEN` 或升级 `torch`。

### ATC 转换为昇腾 OM 格式

ONNX 导出完成后，使用昇腾 ATC 工具编译为专用 OM 格式（在命令行执行）：

```bash
atc --framework=5 \\
    --model='./onnx_export/qwen_merged.onnx' \\
    --output='./om_export/qwen_merged' \\
    --soc_version=Ascend910B3 \\
    --input_shape='input_ids:1,32;attention_mask:1,32' \\
    --log=error
```

**ATC 参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>--framework</code></td>
<td style="text-align: left;"><code>5</code></td>
<td style="text-align: left;">5 = ONNX 框架</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;"><code>Ascend910B3</code></td>
<td style="text-align: left;">必须与实际芯片一致</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;"><code>input_ids:1,32;attention_mask:1,32</code></td>
<td style="text-align: left;">输入名称与形状</td>
</tr>
</table>

**表格解读**：ATC（Ascend Tensor Compiler）是昇腾模型转换工具，将ONNX模型编译为昇腾专用OM格式。`--framework=5`指定输入为ONNX框架（其他：3=Caffe，5=ONNX等）。`--soc_version`必须与实际芯片型号一致（如Ascend910B3），否则编译失败或性能不佳。`--input_shape`指定输入张量的名称和形状，`1,32`表示batch_size=1、序列长度=32，ATC据此优化计算图。

## 8. 部署架构与部署链路

```
训练阶段（实验7.2）                    部署阶段（本实验）
┌─────────────────┐                ┌─────────────────────────┐
│  Qwen1.5-0.5B   │                │  步骤一：权重合并        │
│  (基座模型)     │                │  merge_and_unload()      │
│       +         │       ──→     │  W = W₀ + (α/r)·B·A     │
│  LoRA 微调      │                └────────────┬────────────┘
│  (r=8, α=16)   │                             ↓
└─────────────────┘                ┌─────────────────────────┐
                                      │  步骤二：NPU 部署推理    │
                                      │  PyTorch + torch_npu    │
                                      │  KV Cache + 自回归生成  │
                                      └────────────┬────────────┘
                                                   ↓
                                      ┌─────────────────────────┐
                                      │  步骤三：全面测试验证    │
                                      │  训练集内 + 泛化 + 边界  │
                                      │  微调前后对比            │
                                      └────────────┬────────────┘
                                                   ↓
                                      ┌─────────────────────────┐
                                      │  步骤四/五：进阶部署     │
                                      │  ONNX 导出 → ATC → OM   │
                                      └─────────────────────────┘
```

## 9. 预期结果

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">参考值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">合并耗时</td>
<td style="text-align: left;">~0.5 秒</td>
<td style="text-align: left;">merge_and_unload 速度</td>
</tr>
<tr>
<td style="text-align: left;">推理时延</td>
<td style="text-align: left;">~0.2 秒/问</td>
<td style="text-align: left;">128 token 生成</td>
</tr>
<tr>
<td style="text-align: left;">吞吐量</td>
<td style="text-align: left;">~190 token/s</td>
<td style="text-align: left;">NPU 推理速度</td>
</tr>
<tr>
<td style="text-align: left;">微调后命中率</td>
<td style="text-align: left;">~65%</td>
<td style="text-align: left;">训练集内问题</td>
</tr>
<tr>
<td style="text-align: left;">基座命中率</td>
<td style="text-align: left;">~20%</td>
<td style="text-align: left;">未微调基线</td>
</tr>
<tr>
<td style="text-align: left;">微调提升</td>
<td style="text-align: left;">~45%</td>
<td style="text-align: left;">命中率提升幅度</td>
</tr>
</table>

**表格解读**：预期结果量化了部署各环节的性能指标。合并耗时约0.5秒，因为merge_and_unload只需将低秩矩阵B·A加到基座权重上，计算量很小。推理时延约0.2秒/问（生成128个token），在昇腾910B3 NPU上吞吐量约190 token/s。微调后命中率约65%（训练集内问题），基座命中率约20%（未微调模型对昇腾领域知识了解有限），微调提升约45个百分点——这证明LoRA微调有效且部署正确保留了微调效果。注意实际数值可能因训练随机性略有波动。

## 9.1 测试结果分析与教学说明

> 实际运行中，由于 **Qwen1.5-0.5B-Chat 模型较小（462M 参数）**，且训练数据仅 41 条、训练 5 个 epoch，微调后关键词命中率可能低于预期参考值（~65%），部分训练集内问题可能出现 FAIL。这是**轻量教学配置下的正常现象**，不影响实验的教学价值。

### 为什么结果不够理想？

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">因素</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>模型容量小</strong></td>
<td style="text-align: left;">0.5B 参数的模型知识容量有限，难以通过少量数据注入大量领域知识</td>
</tr>
<tr>
<td style="text-align: left;"><strong>训练数据少</strong></td>
<td style="text-align: left;">仅 41 条问答对，覆盖面窄，模型容易遗忘或混淆</td>
</tr>
<tr>
<td style="text-align: left;"><strong>训练轮数少</strong></td>
<td style="text-align: left;">5 个 epoch 对于大模型微调偏少，知识尚未充分内化</td>
</tr>
<tr>
<td style="text-align: left;"><strong>关键词匹配局限</strong></td>
<td style="text-align: left;">评估方法为硬关键词命中，模型可能用不同表述正确回答但未命中预设关键词</td>
</tr>
</table>

### 为什么作为教学仍然可以？

本实验的核心教学目标是**部署流程**，而非微调精度。从实际结果看，实验已达成以下教学目标：

1. **部署链路完全跑通** — LoRA 权重合并（`merge_and_unload`）、NPU 推理引擎、三层测试框架均正常工作，证明部署流程正确
2. **测试框架有效** — 三层测试体系（训练集内 / 泛化 / 边界）正确识别了模型学到的知识（如昇腾910、Transformer、深度学习等 PASS）和未学到的知识（如 LoRA、达芬奇架构等 FAIL），验证了测试方法的有效性
3. **对比意义达成** — 微调后模型与基座模型的对比差异本身就是教学要点，即使命中率不高，对比过程也展示了如何量化评估微调效果
4. **工程闭环完整** — 学生体验了"训练 → 合并 → 部署 → 测试 → 分析"的完整工程链路，这是工业界模型部署的核心流程

### 如何改善效果？

若需更高的微调命中率，可尝试以下方向（受云沙箱资源限制，本实验不展开）：

- **增大模型**：使用 Qwen1.5-1.8B 或更大模型，知识容量更大
- **增加数据**：扩充训练集至数百条高质量问答对
- **增加轮数**：训练 10~20 个 epoch，充分学习领域知识
- **调整超参**：增大 LoRA 秩 `r`（如 16、32）、调整学习率
- **改进评估**：使用语义相似度或 LLM-as-Judge 替代硬关键词匹配

> **教学结论**：0.5B 模型 + 5 epochs + 41 条数据是典型的轻量教学配置，结果不理想在预期之内。实验的重点在于理解和掌握**大模型部署的完整流程**，而非追求微调精度。

## 10. 课后练习

**第1题**（单选题）LoRA 权重合并的数学公式是？

- A. W = W₀ × B × A
- B. W = W₀ + (α/r)·B·A
- C. W = W₀ - (α/r)·B·A
- D. W = B × A


In [ ]:
q1 = ''
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）合并后模型相比基座+适配器方式的优势是？

- A. 推理零额外延迟，无需 peft 库
- B. 一个基座可挂多个任务
- C. 训练速度更快
- D. 模型体积更小


In [ ]:
q2 = ''
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）KV Cache 的作用是？

- A. 增加模型精度
- B. 缓存已计算的 Key/Value，将推理复杂度从 O(n²) 降为 O(n) per step
- C. 减少模型参数量
- D. 加速模型下载


In [ ]:
q3 = ''
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）预热（Warmup）的必要性是？

- A. 提高模型精度
- B. 触发计算图编译与显存分配，使后续性能测量准确
- C. 减少模型参数
- D. 加速模型下载


In [ ]:
q4 = ''
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）三层测试体系中，"泛化测试"的目的是？

- A. 验证模型学会了训练知识
- B. 验证模型泛化到相关但未见的问题
- C. 测试模型对非领域输入的鲁棒性
- D. 测试模型下载速度


In [ ]:
q5 = ''
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_04 import grade
grade(globals())

## 参考资料

- [PEFT merge_and_unload](https://huggingface.co/docs/peft/main/en/package_reference/peft_model)
- [Qwen1.5-0.5B-Chat](https://huggingface.co/Qwen/Qwen1.5-0.5B-Chat)
- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [ATC 工具使用指南](https://www.hiascend.com/document/detail/zh/canncommercial/80RC3alpha002/inferappdevg/atcapi/atcapi_0001.html)